### Общий алгоритм работы с Optuna

1. Определяем целевую функцию objective, через аргументы она будет получать специальный объект trial. С его помощью можно назначать различные гипермараметры, Например, как в примере ниже, мы задаем x в интервале [-10,10].

2. Далее создаем объект обучения с помощью метода optuna.create_study.

3. Запускаем оптимизацию целевой функции objective на 10 итераций n_trials=10. Происходит 10 вызовов нашей функции с различными параметрами от -10 до 10. Какие именно параметры выбирает optuna будет описано ниже.

In [1]:
import optuna

def objective(trial):
    x = trial.suggest_float('x',-10,10)
    return (x-2) ** 2

study = optuna.create_study()
study.optimize(objective, n_trials=20)

study.best_params

[I 2026-04-06 22:21:28,429] A new study created in memory with name: no-name-cd608109-76de-40de-a412-06aaee0b917a
[I 2026-04-06 22:21:28,448] Trial 0 finished with value: 1.953606264649047 and parameters: {'x': 0.6022853421928183}. Best is trial 0 with value: 1.953606264649047.
[I 2026-04-06 22:21:28,451] Trial 1 finished with value: 72.95814871672506 and parameters: {'x': -6.541554233084577}. Best is trial 0 with value: 1.953606264649047.
[I 2026-04-06 22:21:28,452] Trial 2 finished with value: 59.90626676262842 and parameters: {'x': 9.739913873075619}. Best is trial 0 with value: 1.953606264649047.
[I 2026-04-06 22:21:28,453] Trial 3 finished with value: 0.2991716843657965 and parameters: {'x': 2.546965889581605}. Best is trial 3 with value: 0.2991716843657965.
[I 2026-04-06 22:21:28,455] Trial 4 finished with value: 142.47489756142917 and parameters: {'x': -9.936284914554829}. Best is trial 3 with value: 0.2991716843657965.
[I 2026-04-06 22:21:28,456] Trial 5 finished with value: 59

{'x': 2.5451360962065634}

In [2]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import r2_score
from sklearn.datasets import fetch_california_housing
from lightgbm import LGBMRegressor

RANDOM_STATE = 42

In [3]:
df = fetch_california_housing(as_frame=True)

x = df.data
y = df.target

In [4]:
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.25, random_state=RANDOM_STATE)

### Подбор гиперпараметров c Optuna
Разобъем данные на тренировочную и тестовую часть. На тренировочной части по кросс-валидации подберем гиперпараметры моделей, а затем проверим качество на тестовой части.

In [5]:
def objective_lgbm(trial):
    max_depth = trial.suggest_int("max_depth", 2, 20)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1, log=True)
    n_estimators = trial.suggest_int("n_estimators", 10, 1000)

    score = cross_val_score(LGBMRegressor(max_depth=max_depth, learning_rate=learning_rate, n_estimators=n_estimators),
                            xtrain, ytrain, cv=3, scoring='r2', n_jobs=-1).mean()
    return score


study = optuna.create_study(direction="maximize")
study.optimize(objective_lgbm, n_trials=30, timeout=200)

[I 2026-04-06 22:21:30,383] A new study created in memory with name: no-name-2a89e23d-6af6-410c-a889-16b6606b451f
[I 2026-04-06 22:21:36,569] Trial 0 finished with value: 0.444700773720039 and parameters: {'max_depth': 19, 'learning_rate': 0.00172855962607413, 'n_estimators': 309}. Best is trial 0 with value: 0.444700773720039.
[I 2026-04-06 22:21:41,576] Trial 1 finished with value: 0.835764270402748 and parameters: {'max_depth': 20, 'learning_rate': 0.04117995871063362, 'n_estimators': 343}. Best is trial 1 with value: 0.835764270402748.
[I 2026-04-06 22:21:45,705] Trial 2 finished with value: 0.8235880564866255 and parameters: {'max_depth': 20, 'learning_rate': 0.07452060740734838, 'n_estimators': 90}. Best is trial 1 with value: 0.835764270402748.
[I 2026-04-06 22:21:50,683] Trial 3 finished with value: 0.7899515283928079 and parameters: {'max_depth': 4, 'learning_rate': 0.009384745509623602, 'n_estimators': 628}. Best is trial 1 with value: 0.835764270402748.
[I 2026-04-06 22:21:5

In [6]:
study.best_params

{'max_depth': 14, 'learning_rate': 0.030628682658461137, 'n_estimators': 885}

In [7]:
model = LGBMRegressor(**study.best_params)
model.fit(xtrain, ytrain)

pred = model.predict(xtest)
r2_score(ytest, pred)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000580 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1838
[LightGBM] [Info] Number of data points in the train set: 15480, number of used features: 8
[LightGBM] [Info] Start training from score 2.070349


0.8536224151803036

B Optuna встроена гибкая возможность перебора гиперпараметров.

In [8]:
import sklearn.svm
import sklearn.ensemble
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import optuna

def objective(trial):
    regressor_name = trial.suggest_categorical('regressor', ['SVR', 'RandomForest'])
    
    if regressor_name == 'SVR':
        svr_c = trial.suggest_float('svr_c', 1e-3, 1e3, log=True)
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('reg', sklearn.svm.SVR(C=svr_c))
        ])
    else:
        rf_max_depth = trial.suggest_int('rf_max_depth', 2, 32)
        model = sklearn.ensemble.RandomForestRegressor(
            max_depth=rf_max_depth, 
            n_estimators=10,
            random_state=RANDOM_STATE
        )
    
    model.fit(xtrain, ytrain)
    
    score = model.score(xtest, ytest)
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10) 

print(f"Лучший результат (R2): {study.best_value:.4f}")
print(f"Лучшие параметры: {study.best_params}")

[I 2026-04-06 22:22:52,075] A new study created in memory with name: no-name-f9325983-9aaf-42ac-8972-bed341db4089
[I 2026-04-06 22:24:59,389] Trial 0 finished with value: 0.7531056072868367 and parameters: {'regressor': 'SVR', 'svr_c': 220.37066008791862}. Best is trial 0 with value: 0.7531056072868367.
[I 2026-04-06 22:25:00,207] Trial 1 finished with value: 0.7884286875729237 and parameters: {'regressor': 'RandomForest', 'rf_max_depth': 13}. Best is trial 1 with value: 0.7884286875729237.
[I 2026-04-06 22:25:10,771] Trial 2 finished with value: 0.5684200220464659 and parameters: {'regressor': 'SVR', 'svr_c': 0.00737148019595978}. Best is trial 1 with value: 0.7884286875729237.
[I 2026-04-06 22:25:11,373] Trial 3 finished with value: 0.7894287256513759 and parameters: {'regressor': 'RandomForest', 'rf_max_depth': 15}. Best is trial 3 with value: 0.7894287256513759.
[I 2026-04-06 22:25:21,871] Trial 4 finished with value: 0.7570341286934709 and parameters: {'regressor': 'SVR', 'svr_c':

Лучший результат (R2): 0.7910
Лучшие параметры: {'regressor': 'RandomForest', 'rf_max_depth': 26}
